In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CamembertTokenizer, CamembertModel
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Feature Extraction (RUN ONCE ONLY)

In [ ]:
# Configuration
TEXT_FEATURES = ['full_text', 'source', 'user.location', 'user.description']
NUMERICAL_FEATURES = ['user.statuses_count', 'user.favourites_count', 'user.listed_count']
MAX_LENGTH = 128
BATCH_SIZE = 32

# Check if features already exist
if os.path.exists('features.pkl'):
    print("✓ Features already extracted. Skip to Section 3 to load them.")
else:
    print("Starting feature extraction...")

In [ ]:
# Load raw data
if not os.path.exists('features.pkl'):
    X_train = pd.read_pickle('data_train.pkl')
    X_kaggle = pd.read_pickle('data_kaggle.pkl')
    y_train = pd.read_pickle('y_train.pkl')
    
    # Fill missing values
    for df in [X_train, X_kaggle]:
        for col in TEXT_FEATURES:
            df[col] = df[col].fillna('unknown' if col != 'full_text' else '')
        for col in NUMERICAL_FEATURES:
            df[col] = df[col].fillna(0)
    
    print("✓ Data loaded and cleaned")

In [ ]:
# Load CamemBERT
if not os.path.exists('features.pkl'):
    print("Loading CamemBERT...")
    tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
    camembert = CamembertModel.from_pretrained('camembert-base')
    camembert.to(device)
    camembert.eval()
    
    for param in camembert.parameters():
        param.requires_grad = False
    
    print("✓ CamemBERT loaded")

In [ ]:
# Embedding extraction function
def extract_embeddings(texts, tokenizer, model, device, max_length=128, batch_size=32, desc="Extracting"):
    """Extract CamemBERT embeddings for a list of texts"""
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, 
                           max_length=max_length, return_tensors='pt')
        
        with torch.no_grad():
            outputs = model(
                input_ids=encoded['input_ids'].to(device),
                attention_mask=encoded['attention_mask'].to(device)
            )
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
    
    return np.vstack(embeddings)

In [ ]:
# Extract text embeddings
if not os.path.exists('features.pkl'):
    print("\nExtracting text embeddings...")
    train_embeddings = []
    kaggle_embeddings = []
    
    for feature in TEXT_FEATURES:
        print(f"\nProcessing '{feature}'...")
        train_emb = extract_embeddings(
            X_train[feature].tolist(), tokenizer, camembert, device,
            max_length=MAX_LENGTH, batch_size=BATCH_SIZE, desc=f"  Train"
        )
        kaggle_emb = extract_embeddings(
            X_kaggle[feature].tolist(), tokenizer, camembert, device,
            max_length=MAX_LENGTH, batch_size=BATCH_SIZE, desc=f"  Kaggle"
        )
        train_embeddings.append(train_emb)
        kaggle_embeddings.append(kaggle_emb)
        print(f"  ✓ Shape: {train_emb.shape}")
    
    # Free up GPU memory
    del camembert, tokenizer
    torch.cuda.empty_cache()
    
    print("\n✓ Text embeddings extracted")

In [ ]:
# Process numerical features
if not os.path.exists('features.pkl'):
    print("\nProcessing numerical features...")
    scaler = StandardScaler()
    train_numerical = scaler.fit_transform(X_train[NUMERICAL_FEATURES].values)
    kaggle_numerical = scaler.transform(X_kaggle[NUMERICAL_FEATURES].values)
    print("✓ Numerical features scaled")

In [ ]:
# Combine all features and save
if not os.path.exists('features.pkl'):
    print("\nCombining and saving features...")
    
    X_train_combined = np.hstack(train_embeddings + [train_numerical])
    X_kaggle_combined = np.hstack(kaggle_embeddings + [kaggle_numerical])
    
    # Save everything to a single file
    with open('features.pkl', 'wb') as f:
        pickle.dump({
            'X_train': X_train_combined,
            'y_train': y_train.values,
            'X_kaggle': X_kaggle_combined,
            'kaggle_ids': X_kaggle['challenge_id'].values,
            'scaler': scaler
        }, f)
    
    print(f"\n{'='*70}")
    print("✓ FEATURES SAVED SUCCESSFULLY!")
    print(f"  Train shape: {X_train_combined.shape}")
    print(f"  Kaggle shape: {X_kaggle_combined.shape}")
    print(f"  Total features: {X_train_combined.shape[1]}")
    print(f"  File: features.pkl")
    print(f"{'='*70}")

## 3. Load Features

In [2]:
print("Loading features...")
with open('features.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
y_train = data['y_train']
X_kaggle = data['X_kaggle']
kaggle_ids = data['kaggle_ids']

print(f"✓ Features loaded")
print(f"  Train: {X_train.shape}")
print(f"  Kaggle: {X_kaggle.shape}")
print(f"  Labels: {y_train.shape}")

Loading features...
✓ Features loaded
  Train: (154914, 3075)
  Kaggle: (103380, 3075)
  Labels: (154914,)


## 4. Model Training

In [3]:
# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
HIDDEN_DIMS = [256, 128, 64]
DROPOUT = 0.3

print("Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Architecture: {X_train.shape[1]} -> {' -> '.join(map(str, HIDDEN_DIMS))} -> 2")

Training Configuration:
  Batch size: 32
  Learning rate: 0.001
  Epochs: 15
  Architecture: 3075 -> 256 -> 128 -> 64 -> 2


In [4]:
class TweetDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels) if labels is not None else None
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return self.features[idx], self.labels[idx]
        return self.features[idx]

train_dataset = TweetDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✓ Data loaders ready ({len(train_loader)} batches)")

✓ Data loaders ready (4842 batches)


In [5]:
class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model = FCNN(X_train.shape[1], HIDDEN_DIMS, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print(f"✓ Model initialized ({sum(p.numel() for p in model.parameters()):,} parameters)")

✓ Model initialized (829,634 parameters)


In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for features, labels in tqdm(loader, desc="Training"):
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(loader), correct / total

In [7]:
print(f"\nTraining for {NUM_EPOCHS} epochs...")
print("="*70)

best_acc = 0
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    print(f"Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    scheduler.step(train_loss)
    
    if train_acc > best_acc:
        best_acc = train_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✓ Best model saved ({best_acc*100:.2f}%)")

print(f"\n{'='*70}")
print(f"Training complete! Best accuracy: {best_acc*100:.2f}%")
print(f"{'='*70}")


Training for 15 epochs...

Epoch 1/15


Training: 100%|██████████| 4842/4842 [00:13<00:00, 352.59it/s]


Loss: 0.4584 | Accuracy: 79.13% | LR: 0.001000
✓ Best model saved (79.13%)

Epoch 2/15


Training:  26%|██▌       | 1253/4842 [00:02<00:07, 453.41it/s]


KeyboardInterrupt: 

## 5. Prediction

In [ ]:
print("Loading best model...")
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

kaggle_dataset = TweetDataset(X_kaggle)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=BATCH_SIZE, shuffle=False)

predictions = []
with torch.no_grad():
    for features in tqdm(kaggle_loader, desc="Predicting"):
        if isinstance(features, tuple):
            features = features[0]
        outputs = model(features.to(device))
        _, predicted = torch.max(outputs, 1)
        predictions.extend(predicted.cpu().numpy())

print("✓ Predictions complete")

In [ ]:
output = pd.DataFrame({
    'ID': kaggle_ids,
    'Prediction': predictions
})

output.to_csv('predictions.csv', index=False)

print("\n✓ Predictions saved to 'predictions.csv'")
print(f"\nDistribution:")
print(output['Prediction'].value_counts())
print(f"\nClass 0: {(output['Prediction'] == 0).sum()} samples")
print(f"Class 1: {(output['Prediction'] == 1).sum()} samples")